# Експерименти


In [1]:
!pip install -q pytorch-lightning wandb tokenizers

In [2]:
import os, re, json, pickle, string, random
from typing import List
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

SEED     = 21
MAX_LEN  = 256
PAD, UNK = 0, 1
IGNORE   = -100
N_TRAIN  = 150_000
EPOCHS   = 3
BATCH    = 128

USE_WANDB     = True
WANDB_PROJECT = 'dl-3'
WANDB_ENTITY  = 'yrachynska-kyiv-school-of-economics'
WANDB_SECRET  = 'WANDB_API_KEY'

pl.seed_everything(SEED)
files = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs]
def find(name):
    found = [f for f in files if os.path.basename(f) == name]
    if not found:
        raise FileNotFoundError(f'{name} не знайдено в /kaggle/input. Додайте вихід ноутбука 03 як Input '
                                f'(і зробіть там Save Version). Зараз доступні файли: {[os.path.basename(f) for f in files][:20]}')
    return found[0]

if USE_WANDB:
    try:
        import wandb
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret(WANDB_SECRET))
        list(wandb.Api().projects(WANDB_ENTITY))
        print(f'W&B: підключено, проєкт {WANDB_ENTITY}/{WANDB_PROJECT}')
    except Exception as e:
        print('W&B НЕ підключено:', e, '\nПеревірте назву секрету, WANDB_ENTITY (адреса проєкту wandb.ai/<entity>/dl-3) та Internet on. Далі логуємо лише в CSV.')
        USE_WANDB = False

print('GPU:', torch.cuda.is_available())

Seed set to 21
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yrachynska (kaggle-casmi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B: підключено, проєкт yrachynska-kyiv-school-of-economics/dl-3
GPU: True


In [3]:
with open(find('prepared.pkl'), 'rb') as f:
    prepared = pickle.load(f)

val_text, lab_text, test_text = prepared['val_text'], prepared['lab_text'], prepared['test_text']
print({k: len(prepared['word'][k]['ids']) for k in ['big', 'val', 'lab', 'test']})
print('словник word:', len(prepared['word']['vocab']), '| словник bpe:', prepared['bpe']['vocab_size'])

{'big': 373790, 'val': 2000, 'lab': 26, 'test': 477}
словник word: 179708 | словник bpe: 16000


In [4]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p = 0.0, 0.0, 0.0, 0.0
    for t, pr in zip(y_true, y_pred):
        t = set([hard_processing(s) for s in t])
        pr = set([hard_processing(s) for s in pr])
        tp += len(t & pr); fp += len(pr - t); fn += len(t - pr); p += len(t)
    if tp + fp == 0:
        precision = 1.0 if p == 0 else 0.0
    else:
        precision = tp / (tp + fp)
    if tp + fn == 0:
        recall = 1.0 if p == 0 else 0.0
    else:
        recall = tp / (tp + fn)
    f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return round(f1, 8)

EMOJI = re.compile('[\U0001F000-\U0001FFFF\u2600-\u27BF\uFE0F]')
POST_RULES = set()

def clean_span(s, rules):
    if 'newline' in rules:
        s = s.split('\n')[0]
    if 'emoji' in rules:
        s = EMOJI.sub('', s)
    s = s.strip()
    if 'strip' in rules:
        s = s.strip(' \t,.;:!?-–—"«»“”()')
    if 'letters' in rules and not re.search(r'[^\W\d_]', s):
        return None
    if 'upper' in rules and not any(w[0].isupper() for w in re.findall(r'\w+', s)):
        return None
    return s or None

def decode(text, offsets, probs, thr):
    is_loc = (1 - probs[:, 0]) > thr
    result, start, end = [], None, None
    for t, (t_start, t_end) in enumerate(offsets[:len(probs)]):
        new_loc = is_loc[t] and (probs[t, 1] >= probs[t, 2] or t == 0 or not is_loc[t - 1])
        if new_loc:
            if start is not None:
                result.append(text[start:end])
            start, end = t_start, t_end
        elif is_loc[t] and start is not None:
            end = t_end
        else:
            if start is not None:
                result.append(text[start:end])
            start, end = None, None
    if start is not None:
        result.append(text[start:end])
    if POST_RULES:
        result = [c for c in (clean_span(x, POST_RULES) for x in result) if c]
    return result

def score(probs_list, texts, offsets_list, golds, thr):
    preds = [decode(t, o, p, thr) for t, o, p in zip(texts, offsets_list, probs_list)]
    return comp_metric(golds, preds)

In [5]:
class NERDataset(Dataset):
    def __init__(self, ids, labels):
        self.ids, self.labels = ids, labels
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return torch.tensor(self.ids[i][:MAX_LEN]), torch.tensor(self.labels[i][:MAX_LEN])

def collate(batch):
    ids, labels = zip(*batch)
    return {
        'input_ids': pad_sequence(ids, batch_first=True, padding_value=PAD),
        'labels':    pad_sequence(labels, batch_first=True, padding_value=IGNORE),
        'lengths':   torch.tensor([len(x) for x in ids]),
    }

@torch.no_grad()
def predict_probs(model, ids_list, batch_size=128):
    model.eval()
    chunks = []
    for d, ids in enumerate(ids_list):
        for s in range(0, max(len(ids), 1), MAX_LEN):
            chunks.append((d, s, ids[s:s + MAX_LEN]))
    pieces = {}
    for b in range(0, len(chunks), batch_size):
        batch = chunks[b:b + batch_size]
        x = pad_sequence([torch.tensor(c[2] if c[2] else [PAD]) for c in batch], batch_first=True, padding_value=PAD)
        lengths = torch.tensor([max(len(c[2]), 1) for c in batch])
        probs = torch.softmax(model(x.to(model.device), lengths), dim=-1).cpu().numpy()
        for (d, s, ids), p in zip(batch, probs):
            pieces[(d, s)] = p[:len(ids)]
    out = []
    for d, ids in enumerate(ids_list):
        parts = [pieces[(d, s)] for s in range(0, max(len(ids), 1), MAX_LEN)]
        out.append(np.concatenate(parts) if len(ids) else np.zeros((0, 3), dtype='float32'))
    return out

In [6]:
class NERModel(pl.LightningModule):
    def __init__(self, vocab_size, emb_dim=200, hidden=256, layers=1, bidirectional=True, rnn='lstm',
                 dropout=0.3, lr=2e-3, optimizer='adam', scheduler='onecycle', loc_weight=2.0, emb_matrix=None):
        super().__init__()
        self.save_hyperparameters(ignore=['emb_matrix'])
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        if emb_matrix is not None:
            self.emb.weight.data.copy_(torch.tensor(emb_matrix))
        rnn_cls = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = rnn_cls(emb_dim, hidden, num_layers=layers, batch_first=True, bidirectional=bidirectional,
                           dropout=dropout if layers > 1 else 0)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden * (2 if bidirectional else 1), 3)
        self.register_buffer('class_weight', torch.tensor([1.0, loc_weight, loc_weight]))
        self.val_info = None

    def forward(self, ids, lengths):
        x = self.drop(self.emb(ids))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=ids.size(1))
        return self.out(self.drop(out))

    def _loss(self, batch):
        logits = self(batch['input_ids'], batch['lengths'])
        return nn.functional.cross_entropy(logits.reshape(-1, 3), batch['labels'].reshape(-1),
                                           weight=self.class_weight, ignore_index=IGNORE)

    def training_step(self, batch, batch_idx):
        loss = self._loss(batch)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        self.log('val_loss', self._loss(batch), prog_bar=True)

    def on_before_optimizer_step(self, optimizer):
        norms = [p.grad.norm() ** 2 for p in self.parameters() if p.grad is not None]
        self.log('grad_norm', torch.sqrt(sum(norms)))

    def on_validation_epoch_end(self):
        if self.val_info is None or self.trainer.sanity_checking:
            return
        texts, offsets, golds, ids = self.val_info
        probs = predict_probs(self, ids)
        self.log('val_f1', score(probs, texts, offsets, golds, thr=0.5), prog_bar=True)

    def configure_optimizers(self):
        lr = self.hparams.lr
        if self.hparams.optimizer == 'adam':
            opt = torch.optim.Adam(self.parameters(), lr=lr)
        elif self.hparams.optimizer == 'adamw':
            opt = torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=0.01)
        else:
            opt = torch.optim.SGD(self.parameters(), lr=lr, momentum=0.9)
        steps = self.trainer.estimated_stepping_batches
        if self.hparams.scheduler == 'onecycle':
            sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
        elif self.hparams.scheduler == 'cosine':
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
        else:
            return opt
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sch, 'interval': 'step'}}

In [7]:
from tokenizers import Tokenizer

STREETS_GEN = ['Шевченка', 'Лесі Українки', 'Миру', 'Незалежності', 'Франка', 'Грушевського', 'Героїв України',
               'Гагаріна', 'Бандери', 'Хрещатик', 'Перемоги', 'Мазепи', 'Івасюка', 'Стуса', 'Сагайдачного']
STREETS_ADJ = ['Центральна', 'Соборна', 'Київська', 'Садова', 'Шкільна', 'Польова', 'Солом\'янська', 'Здолбунівська',
               'Покровська', 'Лісова', 'Набережна', 'Миколаївська']
ABBR = ['вул.', 'просп.', 'пров.', 'бул.']
FULL = ['вулиця', 'проспект', 'провулок', 'бульвар']
CITIES = ['Київ', 'Львів', 'Харків', 'Одеса', 'Дніпро', 'Біла Церква', 'Бровари', 'Ірпінь', 'Буча', 'Вишневе',
          'Фастів', 'Бориспіль', 'Чернігів', 'Вінниця', 'Житомир', 'Полтава']
LOCATIVES = ['Києві', 'Львові', 'Харкові', 'Одесі', 'Дніпрі', 'Броварах', 'Бучі', 'Ірпені', 'Фастові', 'Борисполі',
             'Василькові', 'Вишгороді', 'Обухові', 'Яготині', 'Білій Церкві', 'Чернігові', 'Вінниці', 'Житомирі',
             'Полтаві', 'Миронівці', 'Баришівці', 'Березані', 'Оболоні', 'Борщагівці', 'Позняках', 'Осокорках',
             'Теремках', 'Виноградарі', 'Нивках', 'Деснянському районі', 'Дарницькому районі']
TEMPLATES = [('Адреса: {A}.', True), ('Наша адреса — {A}.', True), ('{A} — гарне місце.', True),
             ('Звертайтесь: {C}, {A}.', True), ('Це сталося на {A}.', False),
             ('Магазин розташований на {A}.', False), ('Ми працюємо в місті {C}, {A}.', False),
             ('Це сталося в {L}.', False), ('Нещодавно в {L} знайшли загублений гаманець.', False)]

def make_address(rng, nominative):
    if rng.random() < 0.4:
        prefix = 'вулиця' if (nominative and rng.random() < 0.5) else 'вул.'
        street = rng.choice(STREETS_ADJ)
    else:
        prefix = rng.choice(FULL if (nominative and rng.random() < 0.5) else ABBR)
        street = rng.choice(STREETS_GEN)
    addr = f'{prefix} {street}'
    if rng.random() < 0.6:
        addr += f', {rng.randint(1, 150)}'
    return addr

def tokenize_words(text):
    toks = list(re.finditer(r"\w+|[^\w\s]", text))
    return [m.group() for m in toks], [(m.start(), m.end()) for m in toks]

def make_labels(offsets, spans):
    labels = [0] * len(offsets)
    for s_start, s_end in spans:
        first = True
        for i, (t_start, t_end) in enumerate(offsets):
            if t_start < s_end and t_end > s_start:
                labels[i] = 1 if first else 2
                first = False
    return labels

def make_synthetic(n, kind, rng):
    word_vocab = prepared['word']['vocab']
    if kind == 'bpe':
        if 'tokenizer_json' in prepared['bpe']:
            bpe = Tokenizer.from_str(prepared['bpe']['tokenizer_json'])
        else:
            bpe = Tokenizer.from_file(find('bpe.json'))
    ids_list, labels_list = [], []
    for _ in range(n):
        tpl, nominative = rng.choice(TEMPLATES)
        addr = make_address(rng, nominative)
        values = {'{A}': addr, '{C}': rng.choice(CITIES), '{L}': rng.choice(LOCATIVES)}
        text, spans = '', []
        for part in re.split(r'(\{[ACL]\})', tpl):
            if part in values:
                spans.append((len(text), len(text) + len(values[part])))
                text += values[part]
            else:
                text += part
        if kind == 'word':
            tokens, offsets = tokenize_words(text)
            ids = [word_vocab.get(w.lower(), UNK) for w in tokens]
        else:
            enc = bpe.encode(text)
            offsets = enc.offsets
            ids = [bpe.token_to_id(t) if bpe.token_to_id(t) is not None else UNK for t in enc.tokens]
        ids_list.append(ids)
        labels_list.append(make_labels(offsets, spans))
    return ids_list, labels_list

demo_ids, demo_labels = make_synthetic(3, 'word', random.Random(SEED))
print(demo_ids[0]); print(demo_labels[0])

[1, 70403, 93, 3228, 789, 14]
[1, 2, 0, 0, 0, 0]


In [8]:
if "results" not in globals():
    results = []

def run_experiment(name, kind='word', rnn='lstm', layers=1, bidirectional=True, hidden=256, emb_dim=200,
                   lr=2e-3, optimizer='adam', scheduler='onecycle', dropout=0.3, loc_weight=2.0,
                   synth=0, epochs=EPOCHS, use_pretrained=False, n_train=None, use_lab=0):
    pl.seed_everything(SEED)
    data = prepared[kind]
    vocab_size = len(prepared['word']['vocab']) if kind == 'word' else prepared['bpe']['vocab_size']
    emb_matrix = prepared['word']['emb_matrix'] if (use_pretrained and kind == 'word') else None
    if emb_matrix is not None:
        emb_dim = emb_matrix.shape[1]

    idx = random.Random(SEED).sample(range(len(data['big']['ids'])), min(n_train or N_TRAIN, len(data['big']['ids'])))
    tr_ids    = [data['big']['ids'][i] for i in idx]
    tr_labels = [data['big']['labels'][i] for i in idx]
    if synth:
        s_ids, s_labels = make_synthetic(synth, kind, random.Random(SEED))
        tr_ids += s_ids; tr_labels += s_labels
    if use_lab:
        tr_ids += data['lab']['ids'] * use_lab
        tr_labels += data['lab']['labels'] * use_lab

    keep = [i for i, x in enumerate(tr_ids) if len(x) > 0]
    tr_ids, tr_labels = [tr_ids[i] for i in keep], [tr_labels[i] for i in keep]

    train_dl = DataLoader(NERDataset(tr_ids, tr_labels), batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2)
    val_dl   = DataLoader(NERDataset(data['val']['ids'], data['val']['labels']), batch_size=256, collate_fn=collate)

    model = NERModel(vocab_size, emb_dim, hidden, layers, bidirectional, rnn, dropout, lr, optimizer, scheduler,
                     loc_weight, emb_matrix)
    model.val_info = (val_text, data['val']['offsets'], data['val']['locs'], data['val']['ids'])

    loggers = [pl.loggers.CSVLogger('/kaggle/working/logs', name=name)]
    if USE_WANDB:
        wandb_logger = pl.loggers.WandbLogger(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=name)
        loggers.append(wandb_logger)
    ckpt = pl.callbacks.ModelCheckpoint(dirpath=f'/kaggle/working/ckpt/{name}', monitor='val_f1', mode='max', save_top_k=1)
    trainer = pl.Trainer(max_epochs=epochs, accelerator='auto', devices=1, gradient_clip_val=1.0,
                         logger=loggers, callbacks=[ckpt], log_every_n_steps=50, enable_model_summary=False)
    trainer.fit(model, train_dl, val_dl)

    model = NERModel.load_from_checkpoint(ckpt.best_model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    probs = {'val':  predict_probs(model, data['val']['ids']),
             'lab':  predict_probs(model, data['lab']['ids']),
             'test': predict_probs(model, data['test']['ids'])}

    thrs = np.arange(0.1, 0.91, 0.05)
    val_scores = [score(probs['val'], val_text, data['val']['offsets'], data['val']['locs'], t) for t in thrs]
    thr = float(thrs[int(np.argmax(val_scores))])
    row = {'name': name, 'kind': kind, 'rnn': rnn, 'layers': layers, 'bidir': bidirectional, 'optimizer': optimizer,
           'scheduler': scheduler, 'synth': synth, 'thr': round(thr, 2),
           'val_f1@0.5': score(probs['val'], val_text, data['val']['offsets'], data['val']['locs'], 0.5),
           'val_f1': max(val_scores),
           'lab_f1': None if use_lab else score(probs['lab'], lab_text, data['lab']['offsets'], data['lab']['locs'], thr),
           'lab_in_train': use_lab,
           'LB': None}
    results.append(row)
    with open(f'/kaggle/working/preds_{name}.pkl', 'wb') as f:
        pickle.dump({'probs': probs, 'thr': thr, 'kind': kind}, f)
    if USE_WANDB:
        wandb_logger.experiment.summary.update({k: v for k, v in row.items() if k != 'LB' and v is not None})
        wandb.finish()
    print(row)
    return probs, thr

## Перевірка: чи модель вчиться

In [9]:
small = prepared['word']['big']
dl = DataLoader(NERDataset(small['ids'][:64], small['labels'][:64]), batch_size=32, collate_fn=collate)
tiny = NERModel(len(prepared['word']['vocab']), lr=3e-3, scheduler='none', dropout=0.0)
trainer = pl.Trainer(max_epochs=100, accelerator='auto', devices=1, logger=False, enable_checkpointing=False,
                     enable_model_summary=False, enable_progress_bar=False)
trainer.fit(tiny, dl)
final = float(trainer.callback_metrics['train_loss'])
print('train_loss в кінці:', round(final, 4), '— OK' if final < 0.1 else '— ПІДОЗРІЛО: loss має впасти майже до нуля, шукайте помилку')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the 

train_loss в кінці: 0.0 — OK


## Експерименти

In [10]:
run_experiment('word_lstm_bi', kind='word', rnn='lstm')

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=3` reached.


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▅▅▅▅████████████
grad_norm,█▂▃▂▂▂▂▂▂▂▃▂▁▁▁▂▁▁▁▁▁▂▂▂▁▁▂▁▁▂▂▁▁▁▁▁▂▁▁▁
train_loss,█▄▃▃▂▂▄▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇██
val_f1,▁██
val_loss,█▁▁
bidir,True
epoch,2
grad_norm,0.04183
kind,word
lab_f1,0.42667


{'name': 'word_lstm_bi', 'kind': 'word', 'rnn': 'lstm', 'layers': 1, 'bidir': True, 'optimizer': 'adam', 'scheduler': 'onecycle', 'synth': 0, 'thr': 0.6, 'val_f1@0.5': 0.77047913, 'val_f1': 0.77304262, 'lab_f1': 0.42666667, 'lab_in_train': 0, 'LB': None}


({'val': [array([[9.99815285e-01, 8.49113349e-05, 9.99271724e-05],
          [9.99214292e-01, 6.50447619e-04, 1.35189533e-04],
          [9.99329209e-01, 6.13168522e-04, 5.76794155e-05],
          [9.91757333e-01, 5.78725291e-03, 2.45548808e-03],
          [9.99830484e-01, 9.72054768e-05, 7.22210025e-05],
          [9.99561131e-01, 3.68633511e-04, 7.01313256e-05],
          [9.99949217e-01, 2.64091323e-05, 2.42841234e-05],
          [9.99977469e-01, 7.36944457e-06, 1.51539170e-05],
          [9.99989271e-01, 8.68608095e-06, 2.02024739e-06],
          [9.99255478e-01, 5.84809517e-04, 1.59687246e-04],
          [9.99891043e-01, 7.70137285e-06, 1.01152684e-04],
          [9.99835014e-01, 1.23371559e-04, 4.15393988e-05],
          [9.99872565e-01, 1.21902704e-05, 1.15276409e-04],
          [9.99757946e-01, 1.69098392e-04, 7.29354360e-05],
          [9.99805629e-01, 1.40642253e-04, 5.37097185e-05],
          [9.99635220e-01, 1.31292210e-04, 2.33483603e-04],
          [9.98772562e-01, 1.0372

In [11]:
run_experiment('word_gru_bi', kind='word', rnn='gru')

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=3` reached.


epoch,▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▅▅▅▅▅▅███████████████
grad_norm,█▅▂▃▂▂▂▂▁▂▃▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁
train_loss,█▅▄▃▃▃▂▂▂▂▃▂▂▂▁▁▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
val_f1,▁▇█
val_loss,█▁▁
bidir,True
epoch,2
grad_norm,0.08202
kind,word
lab_f1,0.42667


{'name': 'word_gru_bi', 'kind': 'word', 'rnn': 'gru', 'layers': 1, 'bidir': True, 'optimizer': 'adam', 'scheduler': 'onecycle', 'synth': 0, 'thr': 0.6, 'val_f1@0.5': 0.76787463, 'val_f1': 0.77419355, 'lab_f1': 0.42666667, 'lab_in_train': 0, 'LB': None}


({'val': [array([[9.99935031e-01, 3.12563716e-05, 3.37085767e-05],
          [9.99932170e-01, 5.43063943e-05, 1.35017472e-05],
          [9.99853134e-01, 1.38981151e-04, 7.90042213e-06],
          [9.98592317e-01, 1.27321598e-03, 1.34339381e-04],
          [9.99956727e-01, 3.59208316e-05, 7.33251500e-06],
          [9.99866486e-01, 1.25625011e-04, 7.83175255e-06],
          [9.99966145e-01, 1.61373246e-05, 1.77302136e-05],
          [9.99995589e-01, 1.43310228e-06, 2.96992994e-06],
          [9.99985337e-01, 1.36502458e-05, 9.86210807e-07],
          [9.99909878e-01, 6.83556354e-05, 2.17880606e-05],
          [9.99992251e-01, 7.04893978e-07, 7.07833351e-06],
          [9.99984384e-01, 1.28241372e-05, 2.76416699e-06],
          [9.99995351e-01, 6.55692759e-07, 3.89977549e-06],
          [9.99946833e-01, 4.96641842e-05, 3.43267675e-06],
          [9.99901175e-01, 9.19319573e-05, 6.95181734e-06],
          [9.99903679e-01, 5.10644277e-05, 4.52423010e-05],
          [9.99409080e-01, 5.5368

In [12]:
run_experiment('bpe_lstm_bi', kind='bpe', rnn='lstm')

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=3` reached.


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▅▅▅▅█████████████
grad_norm,█▂▂▁▁▁▁▂▁▁▁▁▁▁▁▂▁▁▁▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▅█▆▅▄▄▃▃▃▂▃▂▂▂▂▁▂▃▂▂▂▃▂▁▂▃▂▂▃▁▂▁▁▂▂▁▂▁▂
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇███
val_f1,▁▇█
val_loss,█▁▁
bidir,True
epoch,2
grad_norm,0.08831
kind,bpe
lab_f1,0.35897


{'name': 'bpe_lstm_bi', 'kind': 'bpe', 'rnn': 'lstm', 'layers': 1, 'bidir': True, 'optimizer': 'adam', 'scheduler': 'onecycle', 'synth': 0, 'thr': 0.75, 'val_f1@0.5': 0.68335635, 'val_f1': 0.71379442, 'lab_f1': 0.35897436, 'lab_in_train': 0, 'LB': None}


({'val': [array([[9.99602377e-01, 3.87499495e-06, 3.93712136e-04],
          [9.99966502e-01, 6.76999298e-06, 2.67377727e-05],
          [9.99482632e-01, 4.56217764e-04, 6.11005235e-05],
          [9.99712765e-01, 1.49982225e-04, 1.37174633e-04],
          [9.97673452e-01, 1.98192731e-03, 3.44656000e-04],
          [9.95184720e-01, 2.47703074e-03, 2.33833399e-03],
          [9.82696414e-01, 1.06926318e-02, 6.61092671e-03],
          [9.93572235e-01, 1.06145180e-05, 6.41712546e-03],
          [9.98830259e-01, 7.61730189e-05, 1.09350542e-03],
          [9.99443591e-01, 3.24869980e-05, 5.23875991e-04],
          [9.99195158e-01, 1.52865396e-04, 6.52006362e-04],
          [9.99504328e-01, 7.15961141e-05, 4.24001453e-04],
          [9.99885082e-01, 7.40643154e-05, 4.08502383e-05],
          [9.98825610e-01, 8.40357388e-04, 3.33957549e-04],
          [9.99621749e-01, 1.65234287e-05, 3.61663348e-04],
          [9.99333680e-01, 1.74384797e-04, 4.91870218e-04],
          [9.99498725e-01, 2.1910

In [13]:
run_experiment('bpe_lstm_bi_2layers', kind='bpe', rnn='lstm', layers=2)

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=3` reached.


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▅▅▅███████████████
grad_norm,▇▂▃▅▆█▄▃▆▃▅▃▄▅▃▄▃▂▆▂▂▃▂▂▅▂▂▂▂▃▁▂▂▄▁▂▃▂▂▃
train_loss,█▆▅▅▅▄▅▄▃▃▃▃▂▂▂▂▂▁▂▁▁▂▁▁▁▁▁▁▁▁▁▂▁▁▁▂▁▁▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇███
val_f1,▁██
val_loss,█▁▁
bidir,True
epoch,2
grad_norm,0.12665
kind,bpe
lab_f1,0.4359


{'name': 'bpe_lstm_bi_2layers', 'kind': 'bpe', 'rnn': 'lstm', 'layers': 2, 'bidir': True, 'optimizer': 'adam', 'scheduler': 'onecycle', 'synth': 0, 'thr': 0.75, 'val_f1@0.5': 0.72724108, 'val_f1': 0.75907716, 'lab_f1': 0.43589744, 'lab_in_train': 0, 'LB': None}


({'val': [array([[9.99958158e-01, 7.42211478e-06, 3.44751970e-05],
          [9.99983549e-01, 8.79953132e-06, 7.63067692e-06],
          [9.99915957e-01, 5.24758252e-05, 3.15367433e-05],
          [9.99960303e-01, 1.42385570e-05, 2.55638115e-05],
          [9.99855638e-01, 8.98790677e-05, 5.44541326e-05],
          [9.99617577e-01, 2.76660227e-04, 1.05680985e-04],
          [9.99005139e-01, 2.26777047e-04, 7.68127153e-04],
          [9.99751866e-01, 2.76712271e-05, 2.20489019e-04],
          [9.99781311e-01, 1.03349485e-04, 1.15406452e-04],
          [9.99698043e-01, 4.24785285e-05, 2.59530265e-04],
          [9.99767005e-01, 2.88397296e-05, 2.04210301e-04],
          [9.99899507e-01, 4.48539176e-05, 5.57018466e-05],
          [9.99925017e-01, 4.34489848e-05, 3.14807185e-05],
          [9.99782145e-01, 1.25909821e-04, 9.20109960e-05],
          [9.99920368e-01, 1.80468360e-05, 6.16807811e-05],
          [9.99758542e-01, 4.18572854e-05, 1.99592076e-04],
          [9.99912262e-01, 1.5456

In [14]:
run_experiment('bpe_lstm_uni', kind='bpe', rnn='lstm', bidirectional=False)

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=3` reached.


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▅▅███████████████
grad_norm,█▂▂▂▁▂▂▂▁▂▂▁▂▁▁▁▂▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁
train_loss,█▆▅▅▅▄▄▃▃▂▂▂▃▂▁▂▂▂▂▂▂▁▂▁▁▃▁▂▁▁▁▁▂▂▁▁▂▁▁▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇██
val_f1,▁▅█
val_loss,█▂▁
bidir,False
epoch,2
grad_norm,0.07594
kind,bpe
lab_f1,0.2716


{'name': 'bpe_lstm_uni', 'kind': 'bpe', 'rnn': 'lstm', 'layers': 1, 'bidir': False, 'optimizer': 'adam', 'scheduler': 'onecycle', 'synth': 0, 'thr': 0.8, 'val_f1@0.5': 0.56832635, 'val_f1': 0.59996231, 'lab_f1': 0.27160494, 'lab_in_train': 0, 'LB': None}


({'val': [array([[9.98956919e-01, 9.93720314e-05, 9.43689083e-04],
          [9.99960780e-01, 1.55881826e-05, 2.36414780e-05],
          [9.99491692e-01, 4.25132836e-04, 8.31390716e-05],
          [9.99869943e-01, 2.98928899e-05, 1.00163830e-04],
          [9.91684318e-01, 7.63429236e-03, 6.81461825e-04],
          [9.95298207e-01, 2.08740844e-03, 2.61441176e-03],
          [9.69943166e-01, 1.90830659e-02, 1.09737357e-02],
          [9.94822025e-01, 3.93785362e-04, 4.78411606e-03],
          [9.96101856e-01, 1.72677659e-03, 2.17134156e-03],
          [9.99114811e-01, 2.02120835e-04, 6.83021732e-04],
          [9.96785998e-01, 1.59062853e-03, 1.62342866e-03],
          [9.93360221e-01, 3.72926285e-03, 2.91063334e-03],
          [9.98397410e-01, 7.28996645e-04, 8.73544661e-04],
          [9.83324230e-01, 1.53763294e-02, 1.29950326e-03],
          [9.99585330e-01, 2.73932856e-05, 3.87315056e-04],
          [9.98902202e-01, 7.79670721e-04, 3.18094098e-04],
          [9.99942422e-01, 4.4905

In [15]:
run_experiment('bpe_lstm_adamw_cosine', kind='bpe', rnn='lstm', optimizer='adamw', scheduler='cosine')

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=3` reached.


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▅▅▅▅████████████████
grad_norm,▂▂▄▆▅▄▃▅▂▂▅▃▃▂▂▁▄▃▄▂▂▆▁▁▃▃▅▃█▄▄▄▃▂▂▃▃▂▄▃
train_loss,█▅█▇▅▅▅▂▄▃▃▃▃▃▃▂▂▃▄▃▃▃▃▁▁▄▂▅▂▃▁▂▁▃▂▁▂▂▂▂
trainer/global_step,▁▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
val_f1,▁█▆
val_loss,█▁▁
bidir,True
epoch,2
grad_norm,0.07801
kind,bpe
lab_f1,0.29333


{'name': 'bpe_lstm_adamw_cosine', 'kind': 'bpe', 'rnn': 'lstm', 'layers': 1, 'bidir': True, 'optimizer': 'adamw', 'scheduler': 'cosine', 'synth': 0, 'thr': 0.8, 'val_f1@0.5': 0.68582375, 'val_f1': 0.71642387, 'lab_f1': 0.29333333, 'lab_in_train': 0, 'LB': None}


({'val': [array([[9.99809086e-01, 2.72514103e-06, 1.88248174e-04],
          [9.99903917e-01, 4.61638629e-05, 4.99423186e-05],
          [9.99594986e-01, 3.07795010e-04, 9.71937043e-05],
          [9.99659300e-01, 1.69073959e-04, 1.71640859e-04],
          [9.95744526e-01, 3.08693713e-03, 1.16850121e-03],
          [9.91529584e-01, 3.36762238e-03, 5.10279881e-03],
          [9.80596185e-01, 2.74261041e-03, 1.66612566e-02],
          [9.98288929e-01, 1.00755542e-05, 1.70092704e-03],
          [9.99664545e-01, 4.76353416e-05, 2.87822768e-04],
          [9.99819696e-01, 3.96578143e-05, 1.40632954e-04],
          [9.99598205e-01, 1.01249083e-04, 3.00494488e-04],
          [9.99582231e-01, 1.35106544e-04, 2.82675668e-04],
          [9.99815047e-01, 1.06352840e-04, 7.86810997e-05],
          [9.98076558e-01, 1.43214723e-03, 4.91213752e-04],
          [9.99342501e-01, 2.61534733e-05, 6.31259405e-04],
          [9.98740613e-01, 1.41689423e-04, 1.11770304e-03],
          [9.99575794e-01, 5.9608

In [16]:
run_experiment('word_lstm_synth', kind='word', rnn='lstm', synth=20_000)

Seed set to 21
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=3` reached.


epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▅▅▅▅▅▅▅▅▅▅▅▅▅███████████
grad_norm,█▂▃▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▂▁▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▂▁▁
train_loss,█▆▃▃▃▂▂▂▂▁▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
val_f1,▁██
val_loss,█▁▁
bidir,True
epoch,2
grad_norm,0.04221
kind,word
lab_f1,0.5


{'name': 'word_lstm_synth', 'kind': 'word', 'rnn': 'lstm', 'layers': 1, 'bidir': True, 'optimizer': 'adam', 'scheduler': 'onecycle', 'synth': 20000, 'thr': 0.65, 'val_f1@0.5': 0.77192982, 'val_f1': 0.78105805, 'lab_f1': 0.5, 'lab_in_train': 0, 'LB': None}


({'val': [array([[9.99960780e-01, 8.83752000e-06, 3.03532779e-05],
          [9.99777734e-01, 2.04939235e-04, 1.73173194e-05],
          [9.99756634e-01, 2.35077168e-04, 8.36183426e-06],
          [9.98889148e-01, 9.08534392e-04, 2.02293653e-04],
          [9.99928832e-01, 6.27286863e-05, 8.45232444e-06],
          [9.99726593e-01, 2.65720300e-04, 7.67230995e-06],
          [9.99990344e-01, 1.66999700e-06, 7.99127065e-06],
          [9.99997735e-01, 5.15426109e-07, 1.74681315e-06],
          [9.99999523e-01, 4.24584044e-07, 2.91053155e-08],
          [9.99979496e-01, 1.96916826e-05, 8.06906257e-07],
          [9.99988079e-01, 4.12552083e-07, 1.15604098e-05],
          [9.99984264e-01, 1.49151228e-05, 8.47541514e-07],
          [9.99997020e-01, 1.28808026e-07, 2.80706513e-06],
          [9.99988675e-01, 1.03568218e-05, 9.41524320e-07],
          [9.99995947e-01, 3.92051425e-06, 1.03367768e-07],
          [9.99992847e-01, 2.85122269e-06, 4.34388630e-06],
          [9.99924779e-01, 7.3362

## Діагностика на lab

In [17]:
def prf(golds, preds):
    tp = fp = fn = 0
    for g, p in zip(golds, preds):
        g = {hard_processing(s) for s in g}; p = {hard_processing(s) for s in p}
        tp += len(g & p); fp += len(p - g); fn += len(g - p)
    return round(tp / (tp + fp), 3) if tp + fp else 0, round(tp / (tp + fn), 3) if tp + fn else 0

def diagnose(name, show_errors=True):
    saved = pickle.load(open(f'/kaggle/working/preds_{name}.pkl', 'rb'))
    data, thr = prepared[saved['kind']], saved['thr']
    for split, texts in [('val', val_text), ('lab', lab_text)]:
        preds = [decode(t, o, p, thr) for t, o, p in zip(texts, data[split]['offsets'], saved['probs'][split])]
        print(f'{split}: (precision, recall) при порозі {thr:.2f} =', prf(data[split]['locs'], preds))
    print('lab F1 за порогами:',
          {round(float(t), 1): score(saved['probs']['lab'], lab_text, data['lab']['offsets'], data['lab']['locs'], t)
           for t in np.arange(0.1, 0.91, 0.1)})
    if show_errors:
        for t, o, p, gold in zip(lab_text, data['lab']['offsets'], saved['probs']['lab'], data['lab']['locs']):
            pred = decode(t, o, p, thr)
            gs, ps = {hard_processing(s) for s in gold}, {hard_processing(s) for s in pred}
            missed = [g for g in gold if hard_processing(g) not in ps]
            extra = [x for x in pred if hard_processing(x) not in gs]
            if missed or extra:
                print('пропущено:', missed, '| зайве:', extra)

diagnose('word_lstm_synth')

val: (precision, recall) при порозі 0.65 = (0.812, 0.752)
lab: (precision, recall) при порозі 0.65 = (0.45, 0.562)
lab F1 за порогами: {0.1: 0.328125, 0.2: 0.38596491, 0.3: 0.42424242, 0.4: 0.41304348, 0.5: 0.47058824, 0.6: 0.46153846, 0.7: 0.48571429, 0.8: 0.47761194, 0.9: 0.45901639}
пропущено: ['Тетерів', 'Фастів', 'Біла Церква', 'Миронівка', 'Яготин', 'Баришівка'] | зайве: []
пропущено: ['Хелма'] | зайве: []
пропущено: [] | зайве: ['Києва', 'України', 'Україні']
пропущено: ['ТЦ Аркадія'] | зайве: ['📍Індустріальний міст']
пропущено: [] | зайве: ['🇵']
пропущено: [] | зайве: ['Київ']
пропущено: ['Шевченківському районі'] | зайве: ['Шевченківському районі оперативники', 'Києва']
пропущено: ['Героїв Дніпра'] | зайве: ['Дніпра']
пропущено: [] | зайве: ['Київському національному']
пропущено: ["вул. Солом'янська, 28"] | зайве: ["вул. Солом'янська, 28\n\nAA5181KM"]
пропущено: [] | зайве: ['Київ', 'Пасажирський', 'Васильків', 'Васильків', 'Центр', 'Київ', 'Пасажирський', 'Боярка', 'Тарасівка

## Постобробка

In [18]:
import itertools
RULES = ['newline', 'emoji', 'strip', 'letters', 'upper']

def eval_rules(name, rule_sets):
    global POST_RULES
    saved = pickle.load(open(f'/kaggle/working/preds_{name}.pkl', 'rb'))
    data, thr = prepared[saved['kind']], saved['thr']
    rows = []
    for rules in rule_sets:
        POST_RULES = set(rules)
        rows.append({'rules': '+'.join(rules) or 'жодного',
                     'val_f1': score(saved['probs']['val'], val_text, data['val']['offsets'], data['val']['locs'], thr),
                     'lab_f1': score(saved['probs']['lab'], lab_text, data['lab']['offsets'], data['lab']['locs'], thr)})
    POST_RULES = set()
    return pd.DataFrame(rows)

rule_sets = [[]] + [[r] for r in RULES] + [['newline', 'emoji', 'letters'], ['newline', 'emoji', 'strip', 'letters'], RULES]
eval_rules('word_lstm_synth', rule_sets)

,rules,val_f1,lab_f1
0,жодного,0.781058,0.500000
1,newline,0.781058,0.527778
2,emoji,0.781058,0.507042
3,strip,0.781281,0.500000
4,letters,0.781504,0.507042
5,upper,0.801385,0.507042
6,newline+emoji+letters,0.781504,0.535211
7,newline+emoji+strip+letters,0.781901,0.535211
8,newline+emoji+strip+letters+upper,0.801793,0.535211


In [19]:
POST_RULES = {'newline', 'emoji', 'letters', 'upper'}
print(sorted(POST_RULES))

['emoji', 'letters', 'newline', 'upper']


## Результати

In [20]:
res = pd.DataFrame(results)
res.sort_values('val_f1', ascending=False)

,name,kind,rnn,layers,bidir,optimizer,scheduler,synth,thr,val_f1@0.5,val_f1,lab_f1,lab_in_train,LB
6,word_lstm_synth,word,lstm,1,True,adam,onecycle,20000,0.65,0.771930,0.781058,0.500000,0,None
1,word_gru_bi,word,gru,1,True,adam,onecycle,0,0.60,0.767875,0.774194,0.426667,0,None
0,word_lstm_bi,word,lstm,1,True,adam,onecycle,0,0.60,0.770479,0.773043,0.426667,0,None
3,bpe_lstm_bi_2layers,bpe,lstm,2,True,adam,onecycle,0,0.75,0.727241,0.759077,0.435897,0,None
5,bpe_lstm_adamw_cosine,bpe,lstm,1,True,adamw,cosine,0,0.80,0.685824,0.716424,0.293333,0,None
2,bpe_lstm_bi,bpe,lstm,1,True,adam,onecycle,0,0.75,0.683356,0.713794,0.358974,0,None
4,bpe_lstm_uni,bpe,lstm,1,False,adam,onecycle,0,0.80,0.568326,0.599962,0.271605,0,None
